# 07 · Upload → Muse Spark → optional SAM 3.1 → Muse Spark

Upload one plant photograph, ask Muse Spark to identify it, then optionally segment plant parts with Meta SAM 3.1 and send each resulting crop to Muse Spark. This is an interactive flow check, not a benchmark or a clinical toxicity assessment.

Run top to bottom. The API key is read from Colab Secrets (`MODEL_API_KEY`) or the environment and is not written to disk. Your image is sent to Meta Model API when you run the Spark or SAM cells. Calls may incur charges.

SAM 3.1 segmentation is an optional section: skip it to use only Muse Spark. The default SAM text prompt is `flower`; change it to `plant` if the image is mostly foliage.

References: [Muse Spark image understanding](https://dev.meta.ai/docs/image-understanding), [SAM 3.1 overview](https://dev.meta.ai/docs/sam/overview), [SAM Python parser](https://dev.meta.ai/docs/sam/client-libraries).

## 1. Install the small API and image dependencies

This notebook uses the hosted Meta Model API for both models. It does not download model weights or require a GPU.

In [ ]:
%pip -q install --upgrade openai Pillow matplotlib numpy meta-sam-parser

## 2. Upload the reviewed toxicity reference JSON first

Upload `plant_toxicity.json` from the project before uploading an image. This cell loads the source-backed reference into this notebook. Toxicity statuses and source records stay local; only the list of supported class names is used to guide Spark’s identity response. Toxicity is looked up locally after Spark returns a visual identity. An unknown or ambiguous match stays unknown.


In [ ]:
import json
import re
from pathlib import Path

try:
    from google.colab import files
    toxicity_upload = files.upload()
    if len(toxicity_upload) != 1:
        raise ValueError(f"Upload exactly one toxicity JSON file; received {len(toxicity_upload)} files.")
    toxicity_filename, toxicity_bytes = next(iter(toxicity_upload.items()))
    toxicity_db = json.loads(toxicity_bytes.decode("utf-8"))
except ImportError:
    toxicity_path = Path(input("Path to plant_toxicity.json: ")).expanduser()
    toxicity_filename = toxicity_path.name
    toxicity_db = json.loads(toxicity_path.read_text(encoding="utf-8"))

if not isinstance(toxicity_db.get("plants"), list) or not toxicity_db["plants"]:
    raise ValueError("The JSON must contain a non-empty 'plants' list.")
if len({plant.get("class_name") for plant in toxicity_db["plants"]}) != len(toxicity_db["plants"]):
    raise ValueError("The toxicity JSON contains duplicate class_name values.")

TOXICITY_SOURCES = toxicity_db.get("sources", {})
LOOKUP_PLANT_LABELS = [plant["class_name"] for plant in toxicity_db["plants"]]

def normalize_plant_name(value):
    """Normalize names only for safe exact or unique whole-phrase matching."""
    return " ".join(re.sub(r"[^a-z0-9]+", " ", str(value).casefold()).split())

def lookup_toxicity(plant_name):
    """Find the unique reviewed entry for a Spark identity; otherwise abstain."""
    unknown = {
        "toxicity_status": "unknown",
        "potential_hazard": None,
        "matched_class": None,
        "scientific_name": None,
        "evidence": [],
        "lookup_note": "No unique reviewed class match; no toxicity status was inferred.",
    }
    if not plant_name or normalize_plant_name(plant_name) in {"unknown", "unidentified", "none"}:
        return {**unknown, "lookup_note": "Spark did not return a specific identity."}

    candidate = normalize_plant_name(plant_name)
    exact, contained = [], []
    for plant in toxicity_db["plants"]:
        names = [plant.get("class_name"), plant.get("scientific_name"), *plant.get("aliases", [])]
        names = {normalize_plant_name(name) for name in names if name}
        if candidate in names:
            exact.append(plant)
        elif any(f" {name} " in f" {candidate} " for name in names if name):
            contained.append(plant)
    matches = exact if exact else contained
    unique = {plant["class_name"]: plant for plant in matches}
    if len(unique) != 1:
        return unknown

    plant = next(iter(unique.values()))
    evidence = []
    for item in plant.get("evidence", []):
        source = TOXICITY_SOURCES.get(item.get("source_id"), {})
        evidence.append({
            **item,
            "source_url": item.get("source_url") or source.get("url", ""),
            "source_title": source.get("title", item.get("source_id", "")),
        })
    return {
        "toxicity_status": plant["toxicity_status"],
        "potential_hazard": plant.get("potential_hazard"),
        "matched_class": plant["class_name"],
        "scientific_name": plant.get("scientific_name"),
        "taxonomic_scope": plant.get("taxonomic_scope"),
        "evidence": evidence,
        "lookup_note": plant.get("review_note") or plant.get("source_qualification"),
    }

print(f"Loaded {len(toxicity_db['plants'])} reviewed plant entries from {toxicity_filename}.")
print("Sources:", ", ".join(source.get("title", source_id) for source_id, source in TOXICITY_SOURCES.items()))


## 3. Configure Meta API access and upload a photograph

In Colab, add `MODEL_API_KEY` under **Secrets** and grant this notebook access. You can also set that environment variable manually in a private runtime. The upload widget accepts one image.

In [ ]:
import base64
import io
import json
import os
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
from openai import AsyncOpenAI, OpenAI

try:
    from google.colab import files, userdata
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Upload exactly one image; received {len(uploaded)} files.")
    image_name, image_bytes = next(iter(uploaded.items()))
    try:
        MODEL_API_KEY = userdata.get("MODEL_API_KEY")
    except Exception:
        MODEL_API_KEY = None
except ImportError:
    image_path = Path(input("Path to an image file: ")).expanduser()
    image_name = image_path.name
    image_bytes = image_path.read_bytes()
    MODEL_API_KEY = os.environ.get("MODEL_API_KEY")

if not MODEL_API_KEY:
    MODEL_API_KEY = os.environ.get("MODEL_API_KEY") or getpass("Meta Model API key: ")
if not MODEL_API_KEY.strip():
    raise ValueError("A Meta Model API key is required to call Muse Spark or SAM.")

image = ImageOps.exif_transpose(Image.open(io.BytesIO(image_bytes))).convert("RGB")
if image.width < 2 or image.height < 2:
    raise ValueError(f"Image dimensions are too small: {image.size}")

MODEL_API_BASE = "https://api.meta.ai/v1"
SPARK_MODEL = "muse-spark-1.3"
SAM_MODEL = "sam-3.1"
spark_client = OpenAI(base_url=MODEL_API_BASE, api_key=MODEL_API_KEY, timeout=120.0, max_retries=2)
sam_client = AsyncOpenAI(base_url=MODEL_API_BASE, api_key=MODEL_API_KEY, timeout=180.0, max_retries=2)

def to_png_bytes(pil_image):
    buffer = io.BytesIO()
    pil_image.save(buffer, format="PNG")
    return buffer.getvalue()

def image_data_url(image_content, mime_type="image/png"):
    return f"data:{mime_type};base64," + base64.b64encode(image_content).decode("ascii")

print(f"Loaded {image_name}: {image.width} × {image.height}")
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis("off");

## 4. Ask Muse Spark about the original image

Muse Spark returns schema-constrained JSON so the fields can be parsed directly. It estimates how many individual flowers are visible and recommends whether segmentation may help. Treat the count and recommendation as a visual estimate; you decide whether to run the optional SAM section. Toxicity is looked up separately from the uploaded, source-linked JSON after the visual response; Spark does not generate toxicity claims.

In [ ]:
PLANT_ASSESSMENT_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "plant_image_assessment",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "plant_identification": {
                    "type": "string",
                    "description": "Most likely plant or flower name, or Unknown."
                },
                "identification_confidence": {
                    "type": "string",
                    "enum": ["high", "medium", "low"]
                },
                "flower_count_estimate": {
                    "type": "integer",
                    "description": "Estimated number of distinct visible blossoms/flowers; use 0 if none are visible."
                },
                "flower_count_confidence": {
                    "type": "string",
                    "enum": ["high", "medium", "low"]
                },
                "flower_count_notes": {"type": "string"},
                "suggest_segmentation": {
                    "type": "boolean",
                    "description": "Whether segmentation would likely help separate multiple flowers, plants, or overlapping parts."
                },
                "segmentation_reason": {"type": "string"},
                "visual_evidence": {
                    "type": "array",
                    "items": {"type": "string"}
                }
            },
            "required": [
                "plant_identification",
                "identification_confidence",
                "flower_count_estimate",
                "flower_count_confidence",
                "flower_count_notes",
                "suggest_segmentation",
                "segmentation_reason",
                "visual_evidence"
            ]
        }
    }
}

PLANT_ASSESSMENT_PROMPT = (
    "Inspect this photograph and estimate the number of distinct visible flowers or blossoms. "
    "Count visible blooms, not leaves or petals. If the number is unclear, provide your best "
    "integer estimate and set flower_count_confidence to low, explaining the ambiguity. Use 0 "
    "when no flowers are visible. Identify the plant only as a visual hypothesis. Set "
    "suggest_segmentation to true when separate flowers/plants or overlapping parts make "
    "individual inspection useful; otherwise set it false. Never infer pet safety or toxicity. "
    "Return only the requested JSON object."
)

def ask_spark_for_assessment(data_url, extra_instruction=""):
    response = spark_client.chat.completions.create(
        model=SPARK_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": PLANT_ASSESSMENT_PROMPT + " " + extra_instruction},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }],
        response_format=PLANT_ASSESSMENT_FORMAT,
        max_completion_tokens=1200,
        reasoning_effort="minimal",
    )
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise RuntimeError("Muse Spark truncated its response at 1200 completion tokens; increase the limit.")
    content = choice.message.content
    if not content:
        raise ValueError("Muse Spark returned an empty assessment.")
    return json.loads(content)

original_data_url = image_data_url(
    image_bytes,
    Image.open(io.BytesIO(image_bytes)).get_format_mimetype() or "image/png",
)
spark_assessment = ask_spark_for_assessment(original_data_url)
print(json.dumps(spark_assessment, indent=2, ensure_ascii=False))
spark_toxicity = lookup_toxicity(spark_assessment["plant_identification"])
print("\nLocal toxicity reference lookup (not generated by Spark):")
print(json.dumps(spark_toxicity, indent=2, ensure_ascii=False))
print("\nSegmentation suggestion:", "Run SAM if useful" if spark_assessment["suggest_segmentation"] else "Probably optional")

## 5. Optional: segment flowers with SAM 3.1

Review `spark_assessment["suggest_segmentation"]` and its reason above. Run this section if you want to inspect the flower masks or test the segmented-image flow. SAM returns one mask per detected match for the short text prompt below. This can produce several flower masks, no masks, or imperfect masks. The parser dependency was installed in section 1.

In [ ]:
from meta_sam_parser import (
    decode_mask_to_raster,
    image_segmentation_format,
    parse_responses_stream,
)

SAM_PROMPT = "flower"  # Try "plant" to target foliage and whole plants.
MAX_SEGMENTS = 6
MIN_MASK_PIXELS = 300

async def segment_uploaded_image(pil_image, prompt=SAM_PROMPT):
    content = to_png_bytes(pil_image)
    data_url = image_data_url(content, "image/png")
    events = await sam_client.responses.create(
        model=SAM_MODEL,
        input=[{
            "type": "message",
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": data_url},
            ],
        }],
        stream=True,
        metadata={"mask_encoding": "one_bit"},
    )
    parsed = parse_responses_stream(events, image_segmentation_format())
    return await parsed.final_result()

segmentation = await segment_uploaded_image(image)
mask_records = [record for record in segmentation.records if hasattr(record, "mask")]
print(f"SAM returned {len(mask_records)} mask(s) for prompt {SAM_PROMPT!r}.")
if segmentation.diagnostics:
    print("Parser diagnostics:", segmentation.diagnostics)

segments = []
for record in mask_records:
    # SAM masks are local to their source-image bounds, so paste each decoded
    # raster back into a full-image canvas before displaying or cropping it.
    raster = np.frombuffer(decode_mask_to_raster(record.mask), dtype=np.uint8)
    local_mask = raster.reshape(record.mask.height, record.mask.width)
    bounds = record.bounds
    left = max(0, int(np.floor(bounds.left)))
    top = max(0, int(np.floor(bounds.top)))
    right = min(image.width, int(np.ceil(bounds.right)))
    bottom = min(image.height, int(np.ceil(bounds.bottom)))
    if right <= left or bottom <= top:
        continue
    local_image = Image.fromarray(local_mask * 255).resize(
        (right - left, bottom - top), resample=Image.Resampling.NEAREST
    )
    mask = np.zeros((image.height, image.width), dtype=bool)
    mask[top:bottom, left:right] = np.asarray(local_image) > 0
    ys, xs = np.where(mask)
    if len(xs) < MIN_MASK_PIXELS:
        continue
    left, right = int(xs.min()), int(xs.max()) + 1
    top, bottom = int(ys.min()), int(ys.max()) + 1
    # Keep a transparent background so Spark sees the segmented object clearly.
    rgba = np.zeros((image.height, image.width, 4), dtype=np.uint8)
    rgba[:, :, :3] = np.asarray(image)
    rgba[:, :, 3] = mask.astype(np.uint8) * 255
    crop = Image.fromarray(rgba).crop((left, top, right, bottom))
    segments.append({"object_id": getattr(record, "object_id", str(len(segments))),
                     "mask": mask, "box": (left, top, right, bottom), "crop": crop})

if not segments:
    print("No usable flower masks found. Try SAM_PROMPT = 'plant' or a clearer image.")
else:
    columns = min(3, len(segments))
    rows = (len(segments) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(5 * columns, 5 * rows), squeeze=False)
    for ax, segment in zip(axes.flat, segments):
        overlay = np.asarray(image).copy()
        overlay[segment["mask"]] = (
            0.55 * overlay[segment["mask"]] + 0.45 * np.array([30, 220, 120])
        ).astype(np.uint8)
        ax.imshow(overlay)
        ax.set_title(f"SAM object {segment['object_id']} · box {segment['box']}")
        ax.axis("off")
    for ax in axes.flat[len(segments):]:
        ax.axis("off")
    plt.tight_layout()

## 6. Send each segmented crop to Muse Spark

This checks the end-to-end path: the original image went to Spark, SAM produced masks, and each cropped masked region now goes to Spark. The notebook then looks each segment identity up in the uploaded JSON. A small number of segments keeps API usage manageable. Transparent pixels mark the area outside the mask.

In [ ]:
if not segments:
    print("Run the SAM cell and inspect at least one mask before running this cell.")
else:
    segment_answers = []
    for index, segment in enumerate(segments[:MAX_SEGMENTS], start=1):
        crop_bytes = to_png_bytes(segment["crop"])
        crop_data_url = image_data_url(crop_bytes, "image/png")
        assessment = ask_spark_for_assessment(
            crop_data_url,
            extra_instruction=(
                "This image is a segmented crop from one detected region. The transparent "
                "background is outside the selected region. Estimate the number of visible "
                "flowers in this crop; keep uncertainty explicit."
            ),
        )
        toxicity_result = lookup_toxicity(assessment["plant_identification"])
        segment_answer = {"object_id": segment["object_id"], **assessment, "toxicity_lookup": toxicity_result}
        segment_answers.append(segment_answer)
        print(f"Segment {index} (SAM object {segment['object_id']}):")
        print(json.dumps(segment_answer, indent=2, ensure_ascii=False))
        print()

    print(f"Sent {len(segment_answers)} of {len(segments)} usable segment(s) to Muse Spark.")

## Interpretation and limitations

Compare Spark's response to the full image with its responses to individual SAM crops and the reference lookup shown for each. A change in answer may reflect the crop, removed context, or an imperfect segmentation. SAM can miss blooms, include background, or split/merge objects. The uploaded table is a limited reference, not a diagnosis or a guarantee of safety. Unknown or unmatched results must not be treated as non-toxic; verify identity and contact a veterinarian if ingestion is suspected.

For a reproducible evaluation across examples, use the project's comparison notebooks and fixed manifests rather than this interactive upload flow.